# 07 – Early Signal Interpretability

The previous notebook showed that market efficiency can be detected surprisingly early.

Using only the first 25% of each market trajectory, a Logistic Regression model achieved:

* Accuracy: 67.5%
* F1 Score: 70.1%

This was only moderately below the full-trajectory model, which achieved:

* Accuracy: 74.7%
* F1 Score: 77.1%

This suggests that the early evolution of market probabilities contains meaningful information about whether a prediction market will ultimately become efficient.

However, the previous notebook focused mainly on predictive performance.

This notebook focuses on interpretability.

---

## Research Question

Which early trajectory features explain the model's ability to detect future market efficiency?

---

## Core Hypothesis

If market efficiency can be predicted early, then some early information-dynamics features should systematically differ between efficient and inefficient markets.

In particular, we investigate whether early versions of the following features contain predictive signal:

* Early Max Drawdown
* Early Probability Range
* Early Realized Volatility
* Early Reversals
* Early Trend
* Early Shannon Entropy

---

## Methodology

This notebook analyzes the early feature datasets constructed in the previous stage:

* early_25
* early_50
* early_75

Each dataset contains trajectory-based features computed using only the first portion of each market's probability history.

The target remains:

$ EfficientMarket = \begin{cases} 1, & \text{if } AbsSurprise \leq \text{median}(AbsSurprise) \
0, & \text{otherwise} \end{cases} $

where:

$ AbsSurprise = |Outcome - FinalProbability| $

---

## Analysis Plan

### 1. Feature Comparison by Efficiency Group

Compare average early features between:

* Efficient markets
* Inefficient markets

This helps identify which early signals differ most clearly across groups.

---

### 2. Logistic Regression Coefficients

Train interpretable Logistic Regression models at each horizon:

* 25%
* 50%
* 75%

Then analyze standardized coefficients to identify the strongest predictors.

---

### 3. Feature Stability Across Horizons

Evaluate whether the same early signals remain important across multiple horizons.

A strong finding would be:

> The same feature is predictive at 25%, 50%, and 75% of market life.

---

### 4. Visual Diagnostics

Use simple visualizations such as:

* Boxplots
* Coefficient plots
* Scatter plots

to understand how early trajectory features separate efficient and inefficient markets.

---

## Expected Contribution

This notebook moves from prediction to explanation.

The goal is not only to show that early market efficiency can be predicted, but also to identify which early information dynamics are responsible for that predictability.

This is important because a useful quantitative signal should be:

* Predictive
* Interpretable
* Stable across horizons
* Economically meaningful

---

## Final Objective

By the end of this notebook, we aim to answer:

What early signals reveal whether a prediction market is likely to become efficient?

If a small number of early trajectory features consistently explain future efficiency, they may serve as the foundation for an early warning system for prediction market quality.


In [6]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

early_25 = pd.read_csv('../data/processed\easly_25.csv')
early_50 = pd.read_csv('../data/processed\easly_50.csv')
early_75 = pd.read_csv('../data/processed\easly_75.csv')



<>:9: SyntaxWarning: invalid escape sequence '\e'
<>:10: SyntaxWarning: invalid escape sequence '\e'
<>:11: SyntaxWarning: invalid escape sequence '\e'
<>:9: SyntaxWarning: invalid escape sequence '\e'
<>:10: SyntaxWarning: invalid escape sequence '\e'
<>:11: SyntaxWarning: invalid escape sequence '\e'
C:\Users\avazq\AppData\Local\Temp\ipykernel_18988\3044729640.py:9: SyntaxWarning: invalid escape sequence '\e'
  early_25 = pd.read_csv('../data/processed\easly_25.csv')
C:\Users\avazq\AppData\Local\Temp\ipykernel_18988\3044729640.py:10: SyntaxWarning: invalid escape sequence '\e'
  early_50 = pd.read_csv('../data/processed\easly_50.csv')
C:\Users\avazq\AppData\Local\Temp\ipykernel_18988\3044729640.py:11: SyntaxWarning: invalid escape sequence '\e'
  early_75 = pd.read_csv('../data/processed\easly_75.csv')


In [8]:
# Comparison of means
features = [
    "early_realized_volatility",
    "early_probability_range",
    "early_trend",
    "early_max_drawdown",
    "early_reversals",
    "early_entropy"]


early_25.groupby("efficient_market")[features].mean()



,early_realized_volatility,early_probability_range,early_trend,early_max_drawdown,early_reversals,early_entropy
efficient_market,,,,,,
0,0.213177,0.311579,-0.001645,0.507146,37.842105,0.410971
1,0.204410,0.348152,-0.001221,0.647933,101.000000,0.288184


In [9]:
early_50.groupby("efficient_market")[features].mean()

,early_realized_volatility,early_probability_range,early_trend,early_max_drawdown,early_reversals,early_entropy
efficient_market,,,,,,
0,0.294397,0.349737,-0.000570,0.548198,84.842105,0.432642
1,0.318784,0.425935,-0.000963,0.732841,257.652174,0.217924


In [10]:
early_75.groupby("efficient_market")[features].mean()

,early_realized_volatility,early_probability_range,early_trend,early_max_drawdown,early_reversals,early_entropy
efficient_market,,,,,,
0,0.363232,0.382263,-0.000325,0.577496,133.631579,0.404880
1,0.447764,0.527783,0.001610,0.834725,518.826087,0.185196


In [12]:
# Mann Whitney Feature by Feature
from scipy.stats import mannwhitneyu
for feature in features:
    efficient = early_25.loc[ early_25["efficient_market"] == 1,feature ].dropna()
    inefficient = early_25.loc[early_25["efficient_market"] == 0,feature].dropna()
    stat, p = mannwhitneyu(efficient,inefficient,alternative="two-sided")
    print(feature, p)

early_realized_volatility 0.8102678545957842
early_probability_range 0.40432041006998476
early_trend 0.7046439716692785
early_max_drawdown 0.17237478356957803
early_reversals 0.0036511560240576092
early_entropy 0.7090183923595964


In [13]:
# training the logistic regression 
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
X = early_25[features]
y = early_25["efficient_market"]
pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler()),("model", LogisticRegression(max_iter=1000))])
pipe.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

In [14]:
coef_25 = pd.Series(pipe.named_steps["model"].coef_[0],index=features).sort_values()
coef_25

early_realized_volatility   -0.315438
early_trend                 -0.176324
early_entropy               -0.072189
early_probability_range      0.003144
early_max_drawdown           0.603672
early_reversals              0.966459
dtype: float64

In [15]:
# early 50 
X = early_50[features]
y = early_50["efficient_market"]
pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler()),("model", LogisticRegression(max_iter=1000))])
pipe.fit(X, y)
coef_50 = pd.Series(pipe.named_steps["model"].coef_[0],index=features).sort_values()
coef_50

early_entropy               -0.436667
early_trend                 -0.376337
early_realized_volatility   -0.125438
early_probability_range      0.009357
early_max_drawdown           0.690284
early_reversals              0.779235
dtype: float64

In [16]:
# early  75
X = early_75[features]
y = early_75["efficient_market"]
pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler()),("model", LogisticRegression(max_iter=1000))])
pipe.fit(X, y)
coef_75 = pd.Series(pipe.named_steps["model"].coef_[0],index=features).sort_values()
coef_75


early_entropy               -0.500177
early_realized_volatility    0.023449
early_probability_range      0.159402
early_trend                  0.443072
early_reversals              0.725123
early_max_drawdown           0.802371
dtype: float64

In [18]:
coef_table = pd.concat([coef_25.rename("25%"), coef_50.rename("50%"),coef_75.rename("75%")],axis=1)
coef_table

,25%,50%,75%
early_realized_volatility,-0.315438,-0.125438,0.023449
early_trend,-0.176324,-0.376337,0.443072
early_entropy,-0.072189,-0.436667,-0.500177
early_probability_range,0.003144,0.009357,0.159402
early_max_drawdown,0.603672,0.690284,0.802371
early_reversals,0.966459,0.779235,0.725123


# Key Insights

##  Early Reversals Are the Strongest Early Signal

The most important feature during the first stages of a market's life is:

$ \text{Early Reversals} $
 

Logistic Regression coefficients:

| Horizon | Coefficient |
|----------|----------:|
| 25% | 0.966 |
| 50% | 0.779 |
| 75% | 0.725 |

### Interpretation

Efficient markets appear to update beliefs frequently.

Rather than moving smoothly toward the final outcome, efficient markets continuously revise information and adjust probabilities.

This suggests that active belief updating is an important characteristic of market efficiency.

---

##  Max Drawdown Becomes Increasingly Important

The importance of drawdown increases consistently across horizons:

| Horizon | Coefficient |
|----------|----------:|
| 25% | 0.604 |
| 50% | 0.690 |
| 75% | 0.802 |

### Interpretation

Efficient markets appear to correct mistakes more aggressively.

Large probability corrections are not necessarily signs of inefficiency. Instead, they may indicate a healthy process of information incorporation.

Markets that rapidly revise incorrect beliefs tend to finish closer to the true outcome.

---

## Entropy Is Negatively Associated with Efficiency

Entropy becomes increasingly important as market life progresses:

| Horizon | Coefficient |
|----------|----------:|
| 25% | -0.072 |
| 50% | -0.437 |
| 75% | -0.500 |

### Interpretation

Efficient markets exhibit lower entropy.

Lower entropy suggests:

- More structured information flow
- Less randomness
- Stronger consensus formation
- Faster convergence toward informative prices

As markets mature, entropy becomes one of the clearest indicators of future efficiency.

---

##  Volatility Is Surprisingly Unimportant

Realized volatility contributes very little to prediction:

| Horizon | Coefficient |
|----------|----------:|
| 25% | -0.315 |
| 50% | -0.125 |
| 75% | 0.023 |

### Interpretation

Contrary to traditional financial intuition, volatility alone does not explain market efficiency in prediction markets.

Large movements are not necessarily informative.

The structure of those movements matters more than their magnitude.

---

## Probability Range Contains Weak Signal

Probability range remains relatively unimportant:

| Horizon | Coefficient |
|----------|----------:|
| 25% | 0.003 |
| 50% | 0.009 |
| 75% | 0.159 |

### Interpretation

Markets do not become efficient simply because they move across a wider probability interval.

The path taken by probabilities appears to be more informative than the total distance traveled.

---

## Efficiency Can Be Detected Very Early

Previous results from the Early Warning System showed:

| Horizon | Accuracy |
|----------|----------:|
| 25% | 67.5% |
| 50% | 67.5% |
| 75% | 67.2% |

Compared with the full-information model:

| Model | Accuracy |
|----------|----------:|
| Full Trajectory | 74.7% |

### Interpretation

Most of the information required to identify market efficiency appears during the first quarter of market life.

This suggests that market quality is revealed surprisingly early.

---

# Final Conclusion

Efficient prediction markets are characterized by:

- Frequent early belief revisions
- Strong error-correction dynamics
- Progressively lower information entropy

Among all features studied, the strongest signals are:

1. **Early Reversals**
2. **Early Max Drawdown**
3. **Early Entropy**

These findings suggest that prediction market efficiency is driven less by volatility and more by the dynamics of information incorporation and belief updating.